# Fazit-Beispielabbildungen

Erzeugt Beispielabbildungen zu visueller Inszenierung, Kommentarraum und methodischer Belastbarkeit.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
__file__ = str(PROJECT_ROOT / "exports" / "notebooks" / "05_fazit_beispielabbildungen_notebook.py")
print(f"Projektordner: {PROJECT_ROOT}")


## Code


In [ ]:
from __future__ import annotations

import csv
import math
import shutil
from pathlib import Path

from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps, ImageStat


ROOT = Path(__file__).resolve().parents[2]
OUT_DIR = ROOT / "exports" / "examples" / "figures"
FRAME_DIR = ROOT / "exports" / "examples" / "frames"
MERGED_CSV = ROOT / "data" / "04_analysis_results" / "visual_features" / "99_AI_AND_REAL_TIKTOK_VIDEOS_all_results_merged.csv"
TOPICS_CSV = ROOT / "comments" / "results" / "04_comment_topics_video_level.csv"
SENTIMENT_CSV = ROOT / "comments" / "results" / "02_comment_sentiment_video_level.csv"
METADATA_CSV = ROOT / "exports" / "examples" / "metadata" / "screenshot_examples_metadata.csv"

TILE_W = 340
TILE_H = 604
LABEL_H = 196
GAP = 24
MARGIN = 38
SECTION_H = 48

BG = (248, 247, 242)
PANEL_BG = (255, 255, 255)
TEXT = (29, 32, 36)
MUTED = (77, 84, 92)
VI = (103, 74, 158)
RI = (32, 116, 116)
ROBUST = (43, 126, 112)
CAUTION = (149, 80, 75)
COMMENT = (74, 92, 155)


def font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf"),
        Path("/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        Path("C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size)
    return ImageFont.load_default()


FONT_SECTION = font(24, True)
FONT_TITLE = font(18, True)
FONT_SMALL = font(13)


def read_csv_dict(path: Path) -> dict[str, dict[str, str]]:
    with path.open(encoding="utf-8-sig", newline="") as fh:
        return {str(row["video_id"]): row for row in csv.DictReader(fh)}


def read_metadata() -> list[dict[str, str]]:
    with METADATA_CSV.open(encoding="utf-8-sig", newline="") as fh:
        return list(csv.DictReader(fh))


def num(value: str | None) -> float | None:
    if value is None or value == "":
        return None
    try:
        number = float(value)
    except ValueError:
        return None
    if math.isnan(number):
        return None
    return number


def fmt(value: str | None, digits: int = 2) -> str:
    number = num(value)
    if number is None:
        return "NA"
    return f"{number:.{digits}f}"


def fmt_int(value: str | None) -> str:
    number = num(value)
    if number is None:
        return "NA"
    return str(int(round(number)))


def clean_label(value: str | None) -> str:
    if not value:
        return "NA"
    return value.replace("_", " ").replace("/", " / ")


def frame_score(path: Path) -> float:
    image = Image.open(path).convert("RGB")
    image.thumbnail((220, 390), Image.Resampling.LANCZOS)
    gray = image.convert("L")
    stat = ImageStat.Stat(gray)
    brightness = stat.mean[0]
    contrast = stat.stddev[0]
    entropy = gray.entropy()
    edge_strength = ImageStat.Stat(gray.filter(ImageFilter.FIND_EDGES)).mean[0]
    saturation = ImageStat.Stat(image.convert("HSV").split()[1]).mean[0]
    return entropy * 16 + contrast * 1.2 + edge_strength + saturation * 0.08 - abs(brightness - 130) * 0.25


def frame_candidates(video_id: str) -> list[Path]:
    return sorted(path for path in FRAME_DIR.glob(f"*_{video_id}.jpeg") if not path.name.startswith("abb_fazit_"))


def select_frame(video_id: str) -> Path:
    candidates = frame_candidates(video_id)
    if not candidates:
        raise FileNotFoundError(f"No selected example frame found for video_id={video_id}")
    return max(candidates, key=frame_score)


def copy_selected_frame(figure_stem: str, pos: int, typ: str, video_id: str, source: Path) -> Path:
    dst = FRAME_DIR / f"{figure_stem}_{pos:02d}_{typ}_{video_id}.jpeg"
    if source.resolve() != dst.resolve():
        shutil.copy2(source, dst)
    return dst


def fit_image(path: Path) -> Image.Image:
    image = Image.open(path).convert("RGB")
    return ImageOps.fit(image, (TILE_W, TILE_H), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))


def draw_wrapped(
    draw: ImageDraw.ImageDraw,
    xy: tuple[int, int],
    text: str,
    max_width: int,
    fnt: ImageFont.ImageFont,
    fill=TEXT,
    max_lines: int = 2,
) -> int:
    words = str(text).split()
    lines: list[str] = []
    line = ""
    for word in words:
        candidate = f"{line} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=fnt)[2] <= max_width or not line:
            line = candidate
        else:
            lines.append(line)
            line = word
    if line:
        lines.append(line)

    x, y = xy
    line_height = draw.textbbox((0, 0), "Ag", font=fnt)[3] + 6
    for visible_line in lines[:max_lines]:
        draw.text((x, y), visible_line, font=fnt, fill=fill)
        y += line_height
    return y


def draw_tile(
    canvas: Image.Image,
    draw: ImageDraw.ImageDraw,
    item: dict[str, str],
    x: int,
    y: int,
    stripe: tuple[int, int, int],
    figure_stem: str,
    pos: int,
) -> dict[str, str]:
    if item.get("source_frame"):
        source = ROOT / item["source_frame"]
        if not source.exists():
            raise FileNotFoundError(source)
    else:
        source = select_frame(item["video_id"])
    canvas.paste(fit_image(source), (x, y))
    copied = copy_selected_frame(figure_stem, pos, item["type"], item["video_id"], source)

    draw.rectangle((x, y + TILE_H, x + TILE_W, y + TILE_H + LABEL_H), fill=PANEL_BG)
    draw.rectangle((x, y + TILE_H, x + 8, y + TILE_H + LABEL_H), fill=stripe)
    label_x = x + 18
    label_y = y + TILE_H + 9
    label_y = draw_wrapped(draw, (label_x, label_y), item["title"], TILE_W - 30, FONT_TITLE, fill=TEXT, max_lines=2)
    for raw_line in item["subtitle"].splitlines()[:4]:
        label_y = draw_wrapped(draw, (label_x, label_y + 3), raw_line, TILE_W - 30, FONT_SMALL, fill=MUTED, max_lines=2)

    row = dict(item)
    row["frame_path"] = str(copied.relative_to(ROOT))
    row["source_frame"] = str(source.relative_to(ROOT))
    return row


def render_two_row_figure(
    path: Path,
    top_title: str,
    bottom_title: str,
    top_items: list[dict[str, str]],
    bottom_items: list[dict[str, str]],
    top_color: tuple[int, int, int],
    bottom_color: tuple[int, int, int],
) -> list[dict[str, str]]:
    columns = max(len(top_items), len(bottom_items))
    width = MARGIN * 2 + columns * TILE_W + (columns - 1) * GAP
    row_h = TILE_H + LABEL_H
    height = MARGIN * 2 + SECTION_H * 2 + row_h * 2 + GAP
    canvas = Image.new("RGB", (width, height), BG)
    draw = ImageDraw.Draw(canvas)

    records: list[dict[str, str]] = []
    y = MARGIN
    draw.text((MARGIN, y), top_title, font=FONT_SECTION, fill=top_color)
    y += SECTION_H
    for idx, item in enumerate(top_items, start=1):
        x = MARGIN + (idx - 1) * (TILE_W + GAP)
        records.append(draw_tile(canvas, draw, item, x, y, top_color, path.stem, idx))

    y += row_h + GAP
    draw.text((MARGIN, y), bottom_title, font=FONT_SECTION, fill=bottom_color)
    y += SECTION_H
    start_pos = len(top_items) + 1
    for idx, item in enumerate(bottom_items, start=1):
        x = MARGIN + (idx - 1) * (TILE_W + GAP)
        records.append(draw_tile(canvas, draw, item, x, y, bottom_color, path.stem, start_pos + idx - 1))

    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path, quality=95)
    return records


def write_metadata(path: Path, rows: list[dict[str, str]]) -> None:
    fieldnames = ["figure", "position", "video_id", "type", "title", "subtitle", "source_frame", "frame_path"]
    with path.open("w", encoding="utf-8-sig", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for idx, row in enumerate(rows, start=1):
            writer.writerow(
                {
                    "figure": path.name.replace("_metadata.csv", ".png"),
                    "position": idx,
                    "video_id": row["video_id"],
                    "type": row["type"],
                    "title": row["title"],
                    "subtitle": row["subtitle"],
                    "source_frame": row["source_frame"],
                    "frame_path": row["frame_path"],
                }
            )


def model_row(video_id: str, merged: dict[str, dict[str, str]]) -> dict[str, str]:
    try:
        return merged[str(video_id)]
    except KeyError as exc:
        raise KeyError(f"Missing video_id={video_id} in merged result CSV") from exc


def topic_row(video_id: str, topics: dict[str, dict[str, str]]) -> dict[str, str]:
    try:
        return topics[str(video_id)]
    except KeyError as exc:
        raise KeyError(f"Missing video_id={video_id} in topic video CSV") from exc


def visual_item(video_id: str, typ: str, title: str, note: str, merged: dict[str, dict[str, str]]) -> dict[str, str]:
    row = model_row(video_id, merged)
    subtitle = "\n".join(
        [
            note,
            f"Stabilität={fmt(row.get('camera_stability__stability_index'))}; Cuts/s={fmt(row.get('cuts__cuts_per_second'))}",
            f"Sättigung={fmt(row.get('saturation__saturation_index'))}; Szene={clean_label(row.get('scene_classification__scene_top1_label'))}",
        ]
    )
    return {"video_id": video_id, "type": typ, "title": title, "subtitle": subtitle}


def comment_item(
    video_id: str,
    typ: str,
    title: str,
    framing: str,
    topics: dict[str, dict[str, str]],
    sentiments: dict[str, dict[str, str]],
) -> dict[str, str]:
    topic = topic_row(video_id, topics)
    sentiment = sentiments.get(str(video_id), {})
    subtitle = "\n".join(
        [
            framing,
            f"Dominantes Topic: {topic['dominant_topic_label']}",
            f"Kommentare={fmt_int(topic.get('comment_count'))}; Sentiment={sentiment.get('dominant_comment_sentiment', 'NA')}",
        ]
    )
    return {"video_id": video_id, "type": typ, "title": title, "subtitle": subtitle}


def eval_item(video_id: str, typ: str, title: str, subtitle: str, source_frame: str) -> dict[str, str]:
    return {"video_id": video_id, "type": typ, "title": title, "subtitle": subtitle, "source_frame": source_frame}


def main() -> None:
    merged = read_csv_dict(MERGED_CSV)
    topics = read_csv_dict(TOPICS_CSV)
    sentiments = read_csv_dict(SENTIMENT_CSV)

    visual_top = [
        visual_item("7537330858357706006", "VI", "VI: farbintensive Performance", "Kontrollierte, stark gesättigte Inszenierung", merged),
        visual_item("7500937479499926806", "VI", "VI: stabile Bühneninszenierung", "Ruhige Kamera und keine Schnitte", merged),
        visual_item("7525293677674007816", "VI", "VI: wiederholbares Set-Design", "Hohe Sättigung, stabile Aufnahme", merged),
    ]
    visual_bottom = [
        visual_item("7586737085831286071", "RI", "RI: starke Schnittdynamik", "Sehr viele schnelle Schnitte", merged),
        visual_item("7204832049985834283", "RI", "RI: situative Reisesequenz", "Viele Szenenwechsel im Alltags-/Reisekontext", merged),
        visual_item("7500687358904290603", "RI", "RI: Aufnahme im Auto", "Situativer Aufnahmeort statt Studio-Setting", merged),
    ]
    visual_rows = render_two_row_figure(
        OUT_DIR / "abb_fazit_visuelle_inszenierung_vi_ri.png",
        "VI-Beispiele: kontrollierte und stabile Inszenierung",
        "RI-Beispiele: Bewegung und situative Aufnahmestrukturen",
        visual_top,
        visual_bottom,
        VI,
        RI,
    )
    write_metadata(OUT_DIR / "abb_fazit_visuelle_inszenierung_vi_ri_metadata.csv", visual_rows)

    comment_top = [
        comment_item("6988896764241693958", "VI", "VI: AI/CGI-Rahmung", "Künstlichkeit wird im Kommentarraum explizit sichtbar", topics, sentiments),
        comment_item("7027954993026190598", "VI", "VI: Robotik-Rahmung", "Kommentare markieren virtuelle Identität", topics, sentiments),
        comment_item("7129589619099847982", "VI", "VI: animierte/edierte Lesart", "Kommentarrahmung fokussiert Gemachtheit", topics, sentiments),
    ]
    comment_bottom = [
        comment_item("7512643920539700526", "RI", "RI: positive Bezugnahme", "Sozial-positive Reaktionen im Kommentarraum", topics, sentiments),
        comment_item("7520049050889080119", "RI", "RI: Interesse und Nähe", "Alltagsnahe bzw. relationale Kommentierung", topics, sentiments),
        comment_item("7571206423640575246", "RI", "RI: Alltags-/Gesundheitsrahmung", "Kommentare beziehen sich stärker auf Lebenswelt und Situation", topics, sentiments),
    ]
    comment_rows = render_two_row_figure(
        OUT_DIR / "abb_fazit_kommentarraum_vi_ri.png",
        "VI-Kommentarraum: Künstlichkeit, AI, CGI und Robotik",
        "RI-Kommentarraum: soziale und alltagsbezogene Reaktionen",
        comment_top,
        comment_bottom,
        COMMENT,
        RI,
    )
    write_metadata(OUT_DIR / "abb_fazit_kommentarraum_vi_ri_metadata.csv", comment_rows)

    robust_top = [
        eval_item(
            "7129589619099847982",
            "VI",
            "Robuster: Schnittdynamik",
            "Modell: sehr wenige Schnitte; menschliche Bewertung: sehr gut passend",
            "exports/examples/frames/abb_kategorie_cuts_v02_01_VI_7129589619099847982.jpeg",
        ),
        eval_item(
            "7014189798315298053",
            "VI",
            "Robuster: Kamerastabilität",
            "Modell: sehr stabil; menschliche Bewertung: sehr gut passend",
            "exports/examples/frames/abb_kategorie_camera_stability_v03_01_VI_7014189798315298053.jpeg",
        ),
        eval_item(
            "7491097821064482070",
            "RI",
            "Robuster: Schärfe",
            "Modell: eher scharf; menschliche Bewertung: sehr gut passend",
            "exports/examples/frames/abb_kategorie_sharpness_v05_01_RI_7491097821064482070.jpeg",
        ),
    ]
    cautious_bottom = [
        eval_item(
            "7465128290244742442",
            "VI",
            "Vorsichtiger: Hautglättung",
            "Proxy-Messung; menschliche Bewertung nur zwischen eher schlecht und teilweise passend",
            "exports/examples/frames/abb_kategorie_skin_smoothness_v02_03_VI_7465128290244742442.jpeg",
        ),
        eval_item(
            "7535881652539510038",
            "VI",
            "Vorsichtiger: Beauty Score",
            "Gesichtsbasierter Score; menschliche Bewertung eher schlecht passend",
            "exports/examples/frames/abb_kategorie_beauty_score_v02_03_VI_7535881652539510038.jpeg",
        ),
        eval_item(
            "7586737085831286071",
            "RI",
            "Vorsichtiger: Körperpose",
            "Pose-Proxy; menschliche Bewertung sehr schlecht passend",
            "exports/examples/frames/abb_kategorie_body_pose_v01_03_RI_7586737085831286071.jpeg",
        ),
    ]
    eval_rows = render_two_row_figure(
        OUT_DIR / "abb_fazit_methodische_belastbarkeit.png",
        "Belastbarere Modelloutputs",
        "Vorsichtiger zu interpretierende Modelloutputs",
        robust_top,
        cautious_bottom,
        ROBUST,
        CAUTION,
    )
    write_metadata(OUT_DIR / "abb_fazit_methodische_belastbarkeit_metadata.csv", eval_rows)

    print("Created conclusion figures:")
    for name in [
        "abb_fazit_visuelle_inszenierung_vi_ri.png",
        "abb_fazit_kommentarraum_vi_ri.png",
        "abb_fazit_methodische_belastbarkeit.png",
    ]:
        print(f"- {OUT_DIR / name}")



## Ausführung


In [ ]:
main()
